<a href="https://colab.research.google.com/github/Vandana-kannan/AI-agents/blob/main/ollema_calculator_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 111 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (11.8 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
!ollama --version

In [8]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

response = requests.get("http://localhost:11434/api/tags")

print("Server status:", response.status_code)
print("Ollama server is running!")

Server status: 200
Ollama server is running!


In [9]:
!ollama pull qwen2.5:3b


In [10]:
!pip install -U langchain-core langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 14.2 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-ollama]


In [11]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

response = llm.invoke("Explain artificial intelligence in simple words.")

print(response.content)

Sure! Artificial Intelligence (AI) is like a smart helper that can learn and do things that usually require human intelligence. Here’s a simple way to think about it:

1. **Learning**: AI can learn from data, just like you learn from books or experiences. It can recognize patterns and make decisions based on that learning.

2. **Doing Tasks**: AI can do tasks that are repetitive or complex, like playing chess, driving a car, or diagnosing diseases. It can do these tasks faster and more accurately than humans in some cases.

3. **Understanding**: AI can understand and interpret information, just like you understand and interpret human language. It can answer questions, recognize faces, or even understand emotions in some cases.

4. **Improving**: As AI learns more, it gets better at its tasks. It’s like how you get better at a game the more you play it.

5. **Making Decisions**: AI can make decisions based on the information it has learned. It’s like when you decide what to wear based o

In [12]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers and return the result."""
    return a * b

print(multiply.invoke({"a": 25, "b": 10}))

250


In [13]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# Give the model access to the calculator
tools = [multiply]

llm_with_tools = llm.bind_tools(tools)

# Store tools so Python can execute them
tool_map = {tool.name: tool for tool in tools}


def run_agent(question):

    messages = [
        SystemMessage(
            content=(
                "You are a helpful AI assistant. "
                "Use the calculator tool whenever multiplication "
                "is needed. Give clear and simple answers."
            )
        ),
        HumanMessage(content=question)
    ]

    # Allow multiple tool calls if necessary
    for step in range(5):

        response = llm_with_tools.invoke(messages)

        # Add the model response to conversation history
        messages.append(response)

        # If no tool is requested, return the answer
        if not response.tool_calls:
            return response.content

        # Execute the tools requested by the model
        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            if tool_name not in tool_map:
                raise ValueError(f"Unknown tool: {tool_name}")

            print("Agent is using tool:", tool_name)
            print("Tool inputs:", tool_args)

            result = tool_map[tool_name].invoke(tool_args)

            print("Tool result:", result)

            # Send the tool result back to the model
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

    return "Agent stopped because it reached the maximum number of steps."

In [14]:
answer = run_agent("What is an AI agent?")
print(answer)

An AI agent is a software program designed to simulate intelligent behavior. These agents can be programmed to perform tasks, learn from experience, and interact with their environment. They are used in various applications such as robotics, gaming, and automation. Would you like to know more about a specific aspect of AI agents?


In [15]:
answer = run_agent("Calculate 250 multiplied by 48.")
print(answer)

Agent is using tool: multiply
Tool inputs: {'b': 48, 'a': 250}
Tool result: 12000
The result of multiplying 250 by 48 is 12000.


In [16]:
answer = run_agent(
    "I purchased 5 products. Each costs 350 rupees. What is the total?"
)

print(answer)

Agent is using tool: multiply
Tool inputs: {'b': 350, 'a': 5}
Tool result: 1750
The total cost for purchasing 5 products, each costing 350 rupees, is 1750 rupees.


In [17]:
answer = run_agent(
    "what all the features of the AI agent"
)

print(answer)

As an AI assistant, I can provide information, answer questions, perform calculations, and help with a variety of tasks. I can also engage in conversations on a wide range of topics. However, I don't have features in the traditional sense like a physical agent might. Instead, I have capabilities and functions that I can use to assist you. How can I help you today with these capabilities?


In [18]:
answer = run_agent(
    "what are the different AI models are now days in market"
)

print(answer)

There are numerous AI models available today, each designed for specific tasks such as natural language processing, computer vision, and more. Some of the most popular AI models include:

- BERT (Bidirectional Encoder Representations from Transformers)
- GPT (Generative Pre-trained Transformer)
- T5 (Text-to-Text Transfer Transformer)
- RoBERTa (a refined version of BERT)
- DistilBERT (a smaller, faster version of BERT)
- ALBERT (All Rights Reserved Model with Lm Adaptive)
- XLM-R (Cross-lingual/ Multilingual BERT)
- MobileBERT (optimized for mobile devices)
- ELECTRA (Efficiently Enhanced Transformer-based Representations)
- ConvBert (Convolutional BERT)
- PEGASUS (Parallel End-to-End Speech-to-Text)
- ALBERT (All Rights Reserved Model with Lm Adaptive)
- and many more...

These models are continuously being developed and improved by researchers and companies around the world. Each model has its own strengths and is suited for different applications.
